# Chuẩn bị dữ liệu và xây dựng cơ sở tri thức (Knowledge Base)

Tệp notebook này thực hiện nhúng (embedding) dữ liệu bài viết y tế bằng mô hình `paraphrase-multilingual-mpnet-base-v2` và lưu trữ vào cơ sở dữ liệu vector Qdrant để phục vụ truy xuất (retrieval).


# Khai báo thư viện



In [ ]:
!pip install qdrant-client sentence-transformers pandas langchain langchain-community langchain-qdrant langchain-huggingface bitsandbytes accelerate pyarrow datasets

import pandas as pd
from datasets import load_dataset
import json
from sentence_transformers import SentenceTransformer
from langchain_qdrant import Qdrant
from qdrant_client import QdrantClient
from qdrant_client.http.models import VectorParams, Distance, PointStruct
from langchain_core.documents import Document
from langchain_community.embeddings import SentenceTransformerEmbeddings

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.7/306.7 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88

In [ ]:
import os
import pyarrow.parquet as pq
import numpy as np
import torch
import time

# Lưu dữ liệu vào Qdrant (chỉ chạy 1 lần, k cần chạy lại)

In [ ]:
vinmec_file_path = "/content/vinmec_article_subtitle.parquet"
vinmec_df = pd.read_parquet(vinmec_file_path)
print(f"Số dòng trong vinmec_article_subtitle: {len(vinmec_df)}")

# Tạo danh sách Document từ cột content
documents = [
    Document(page_content=row["content"])
    for _, row in vinmec_df.iterrows()
]

Số dòng trong vinmec_article_subtitle: 163186


In [ ]:
# Tải mô hình embedding
embedder = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

# Nhúng vector
texts = [doc.page_content for doc in documents]
embeddings = embedder.encode(texts, show_progress_bar=True, batch_size=64)
print(f"Đã nhúng {len(embeddings)} vector, kích thước mỗi vector: {len(embeddings[0])}")

# Kết nối Qdrant
client = QdrantClient(
    url="***", # đã ẩn
    api_key="***", # đã ẩn
)
collection_name = "vietnamese_healthcare_knowledge"

# Xóa collection cũ và tạo mới
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)
client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)

# Lưu dữ liệu theo lô
upload_batch_size = 100
for i in range(0, len(embeddings), upload_batch_size):
    batch_points = [
        PointStruct(
            id=i + idx,
            vector=embedding.tolist(),
            payload={"content": doc.page_content}
        )
        for idx, (embedding, doc) in enumerate(zip(embeddings[i:i + upload_batch_size],
                                                  documents[i:i + upload_batch_size]))
    ]
    client.upsert(collection_name=collection_name, points=batch_points)
    print(f"Đã lưu lô {i // upload_batch_size + 1}: {len(batch_points)} điểm")

# Kiểm tra số điểm
point_count = client.count(collection_name=collection_name).count
print(f"Số điểm trong Qdrant: {point_count}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2550 [00:00<?, ?it/s]

KeyboardInterrupt: 

# Tạo ngữ cảnh cho medical_QA (chỉ chạy 1 lần, k cần chạy lại)

In [ ]:
medical_qa_file_path = "/content/medical_qa.parquet"
medical_qa_df = pd.read_parquet(medical_qa_file_path)
print(f"Số dòng trong medical_qa : {len(medical_qa_df)}")

# Kết nối vector store
embeddings = SentenceTransformerEmbeddings(model_name="paraphrase-multilingual-mpnet-base-v2") #mô hình nhúng đa ngôn ngữ
vector_store = Qdrant(
    client=client,
    collection_name=collection_name,
    embeddings=embeddings,
    content_payload_key="content"
)

Số dòng trong medical_qa (trước lọc): 10015


In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
# Tải hai mô hình cross-encoder
cross_encoder_ms_marco = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2', device='cuda') #cái này k dùng
cross_encoder_bge = CrossEncoder('BAAI/bge-reranker-base', device='cuda')

config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [ ]:
#TẠO NGỮ CẢNH VỚI RERANKING
output_file = "/content/fine_tune_data_10k_reranked.jsonl"
no_context_count = 0  # Biến đếm số dòng không tìm thấy ngữ cảnh
batch_size = 100  # Xử lý 100 dòng mỗi lần
total_rows = len(medical_qa_df)

# Mở file JSONL để lưu dần
with open(output_file, "w", encoding="utf-8") as f:
    for batch_start in range(0, total_rows, batch_size):
        batch_end = min(batch_start + batch_size, total_rows)
        batch_df = medical_qa_df.iloc[batch_start:batch_end]
        print(f"\nXử lý batch {batch_start // batch_size + 1}: Dòng {batch_start + 1} đến {batch_end}")
        start_time = time.time()

        for idx, row in batch_df.iterrows():
            question = str(row["title"])
            answer = str(row["content"])

            # Truy xuất top 20 tài liệu từ Qdrant có độ tương đồng>=0.3
            results = vector_store.similarity_search_with_score(question, k=20, score_threshold=0.3)

            # Chuẩn bị dữ liệu cho reranking
            docs = [doc.page_content for doc, score in results]
            scores_qdrant = [score for doc, score in results]

            # Reranking với bge-reranker-base
            if docs:
                pairs = [[question, doc] for doc in docs]
                scores_bge = cross_encoder_bge.predict(pairs)

                # Lọc tài liệu có điểm reranking >= 0.3
                valid_indices = [i for i, score in enumerate(scores_bge) if score >= 0.3]
                ranked_indices_bge = sorted(valid_indices, key=lambda i: scores_bge[i], reverse=True)[:3]  # Lấy top 3
                ranked_docs_bge = [docs[i] for i in ranked_indices_bge]
                ranked_scores_bge = [float(scores_bge[i]) for i in ranked_indices_bge]  # Chuyển sang float
                ranked_scores_qdrant_bge = [scores_qdrant[i] for i in ranked_indices_bge]
                context_bge = "\n".join(ranked_docs_bge) if ranked_docs_bge else "Không tìm thấy ngữ cảnh phù hợp."
            else:
                context_bge = "Không tìm thấy ngữ cảnh phù hợp."
                ranked_scores_qdrant_bge = []
                ranked_scores_bge = []

            # Đếm dòng không tìm thấy ngữ cảnh
            if context_bge == "Không tìm thấy ngữ cảnh phù hợp.":
                no_context_count += 1

            # Lưu dòng vào JSONL
            item = {
                "question": question,
                "context": context_bge,
                "answer": answer
            }
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

            # In tiến độ mỗi 1000 dòng
            if (idx + 1) % 1000 == 0:
                print(f"Đã xử lý {idx + 1} dòng. Không tìm thấy ngữ cảnh: {no_context_count}")

        # In thời gian batch
        batch_time = time.time() - start_time
        print(f"Batch hoàn thành trong {batch_time:.2f} giây.")

        # Xóa bộ nhớ GPU
        torch.cuda.empty_cache()

print(f"\nSố dòng không tìm thấy ngữ cảnh phù hợp: {no_context_count} / {total_rows}")
print(f"Đã lưu tập fine-tuning ({total_rows} dòng) vào {output_file}")


Xử lý batch 1: Dòng 1 đến 100
Batch hoàn thành trong 70.74 giây.

Xử lý batch 2: Dòng 101 đến 200
Batch hoàn thành trong 70.11 giây.

Xử lý batch 3: Dòng 201 đến 300
Batch hoàn thành trong 71.01 giây.

Xử lý batch 4: Dòng 301 đến 400
Batch hoàn thành trong 70.60 giây.

Xử lý batch 5: Dòng 401 đến 500
Batch hoàn thành trong 70.37 giây.

Xử lý batch 6: Dòng 501 đến 600
Batch hoàn thành trong 71.60 giây.

Xử lý batch 7: Dòng 601 đến 700
Batch hoàn thành trong 70.08 giây.

Xử lý batch 8: Dòng 701 đến 800
Batch hoàn thành trong 70.11 giây.

Xử lý batch 9: Dòng 801 đến 900
Batch hoàn thành trong 70.00 giây.

Xử lý batch 10: Dòng 901 đến 1000
Đã xử lý 1000 dòng. Không tìm thấy ngữ cảnh: 259
Batch hoàn thành trong 70.99 giây.

Xử lý batch 11: Dòng 1001 đến 1100
Batch hoàn thành trong 70.65 giây.

Xử lý batch 12: Dòng 1101 đến 1200
Batch hoàn thành trong 70.70 giây.

Xử lý batch 13: Dòng 1201 đến 1300
Batch hoàn thành trong 70.35 giây.

Xử lý batch 14: Dòng 1301 đến 1400
Batch hoàn thành trong

**Hết**